In [ ]:
from pathlib import Path
import os
import glob

import pandas as pd
import geopandas as gpd                          
import numpy as np
import xarray as xr
from scipy import sparse


import matplotlib.pyplot as plt 
import cartopy.crs as ccrs
from matplotlib.colors import ListedColormap
from matplotlib.colors import BoundaryNorm
from cartopy.io import shapereader   



from climada.hazard import TCTracks
from climada.hazard import Centroids
from climada.hazard import TropCyclone
from climada.util.plot import plot_from_gdf


In [ ]:
############################################################################################################
# below we use TC Track data (txt files)
############################################################################################################

In [ ]:
tracks_list = sorted(glob.glob("/lfs/home/yanlan/climada/tc-risk/tc_track_data/TCtrack.slp.4C_*_48.txt"))

In [ ]:
years = [int(os.path.basename(f).split("_")[1][:4]) for f in tracks_list]
num_of_years = (max(years) - min(years) + 1)

In [ ]:
histogram_x, histogram_y = np.unique(np.array(years), return_counts=True)
plt.bar(histogram_x, histogram_y)
plt.ylabel("#")
plt.xlabel("Year")

In [ ]:
############################################################################################################
# IBTracks
############################################################################################################

In [ ]:
# trigger the latest downlaod with a random storm_id
_ = TCTracks.from_ibtracs_netcdf(storm_id="1979174N07143")

In [ ]:
ibtracks = xr.open_dataset("/lfs/home/yanlan/climada/data/IBTrACS.ALL.v04r01.nc")
ibtracks # Coordinates ≈ Data variables

In [ ]:
############################################################################################################
# IBTracks limited to WP AND year >= 1979
############################################################################################################

In [ ]:
min_year = min(years) # 1979
wp_mask = (ibtracks["basin"] == b"WP").any(dim="date_time")
year_mask = ibtracks["season"] >= min_year

ibtracks_wp = ibtracks.isel(storm=wp_mask & year_mask)
ibtracks_wp

In [ ]:
############################################################################################################
# select the 197 historicals 
############################################################################################################

In [ ]:
from collections import defaultdict

storm_lookup = defaultdict(list)

for i in range(ibtracks_wp.sizes["storm"]):

    year = int(ibtracks_wp["season"][i].item())
    name = ibtracks_wp["name"][i].item().decode()
    
    storm_lookup[(year, name)].append(i)

In [ ]:
assert ibtracks_wp.sizes["storm"] == sum(len(v) for v in storm_lookup.values())

In [ ]:
############################################################################################################
# check naming errors
############################################################################################################

In [ ]:
for f in tracks_list:
    year_number_name = Path(f).stem.split("_")[1] # 197901ELLIS

    year = int(year_number_name[:4])
    name = year_number_name[6:]

    if (year, name) not in storm_lookup:
        print("Missing in IBTracks:", (year, name))
        print("Possible IBTrACS names:", ", ".join(n for y, n in storm_lookup if y == year))
        print("-" * 100)

In [ ]:
name_fixes = {
              "BELLY": "BETTY",
              "BRENDA": "BRENDAN"
             }

idx = []

for f in tracks_list:
    
    year_number_name = Path(f).stem.split("_")[1] # 197901ELLIS
    year = int(year_number_name[:4])
    name = year_number_name[6:] 
    name = name_fixes.get(name, name) # neat!


    with open(f) as example_file:
        all_rows = example_file.readlines()

    first_row = all_rows[0].split()
    last_row = all_rows[-1].split()

    time_first = np.datetime64(
        f"{first_row[1]}-{first_row[2]}-{first_row[3]}T{first_row[4]}:00")
    
    time_last = np.datetime64(
        f"{last_row[1]}-{last_row[2]}-{last_row[3]}T{last_row[4]}:00")


    if len(storm_lookup[(year, name)]) > 1:

        storm_lookup[(year, name)] = [i for i in storm_lookup[(year, name)]
                                      if (
                                        ibtracks_wp["time"][i].dropna("date_time").values.min() <= time_first
                                        and
                                        ibtracks_wp["time"][i].dropna("date_time").values.max() >= time_last
                                        )
                                     ]


    
    assert len(storm_lookup[(year, name)]) == 1
       
    idx.extend(storm_lookup[(year, name)])

In [ ]:
assert len(idx) == len(tracks_list)

In [ ]:
ibtracks_wp_197 = ibtracks_wp.isel(storm=idx) 

In [ ]:
############################################################################################################
# "ib_all_tracks_haz"
############################################################################################################

In [ ]:
ibtracks_197_id = ibtracks_wp_197.sid.values.astype(str).tolist()
ib_all_tracks = TCTracks.from_ibtracs_netcdf(storm_id=ibtracks_197_id)
############################################################################################################
# centroids = from WRF
lat_min, lat_max = 20.0, 29.0
lon_min, lon_max = 117.0, 126.0

files_list = sorted(glob.glob("/lfs/home/yanlan/climada/tc-risk/PGW4K.v230926/wrfout_d01_*_48.nc"))

with xr.open_dataset(files_list[0]) as ref:

    # mask is of xarray.DataArray
    mask = (
            (ref["XLAT"].isel(Time=0)>=lat_min) & (ref["XLAT"].isel(Time=0)<=lat_max) &
            (ref["XLONG"].isel(Time=0)>=lon_min) & (ref["XLONG"].isel(Time=0)<=lon_max)
           ) 

    centroids = Centroids(
                          lat=ref["XLAT"].isel(Time=0).values.ravel()[mask.values.ravel()],
                          lon=ref["XLONG"].isel(Time=0).values.ravel()[mask.values.ravel()]
                         )
############################################################################################################
# other Hazard variables 
# ...
############################################################################################################
# other Hazard variables  
# ...

In [ ]:
ib_all_tracks_haz = TropCyclone.from_tracks(tracks=ib_all_tracks, 
                                            centroids=centroids, # centroids = from WRF
                                            model='H1980', 
                                            model_kwargs={"gradient_to_surface_winds": 0.9},
                                            intensity_thres=0) 

In [ ]:
ib_all_tracks_haz.check()

In [ ]:
############################################################################################################
# exceedance_intensities
############################################################################################################

In [ ]:
cmap = ListedColormap(["#fff7bc",  # 0–10   pale yellow
                       "#fee391",  # 10–20  yellow
                       "#fec44f",  # 20–30  amber
                       "#fe9929",  # 30–35  orange
                       "#ec7014",  # 35–40  dark orange
                       "#cc4c02",  # 40–45  burnt orange
                       "#a63603",  # 45–50  brick
                       "#7f2704",  # 50–55  dark brick
                       "#67000d"]) # 55–60  deep red
                      
cmap.set_bad("lightgray")  

cmap.set_over("#3f0000") 

levels = [0, 10, 20, 30, 35, 40, 45, 50, 55, 60]
 
norm = BoundaryNorm(levels, ncolors=cmap.N, clip=False)

ib_all_tracks_haz.intensity = sparse.csc_matrix(ib_all_tracks_haz.intensity)
exceedance_intensities, label, column_label = ib_all_tracks_haz.local_exceedance_intensity([10, 20, 30, 50]) # given return periods
ib_all_tracks_haz.intensity = sparse.csr_matrix(ib_all_tracks_haz.intensity)

# mask People's Republic of China as nan
shapefile = shapereader.natural_earth(resolution="10m", category="cultural", name="admin_0_countries")

world = gpd.read_file(shapefile)

china = world[world.NAME == "China"].geometry.union_all()

china_mask = exceedance_intensities.geometry.within(china)

value_cols = [col for col in exceedance_intensities.columns if col != "geometry"]

exceedance_intensities.loc[china_mask, value_cols] = np.nan

In [ ]:
############################################################################################################
# plot exceedance_intensities
############################################################################################################

In [ ]:
axes = plot_from_gdf(exceedance_intensities,
                     colorbar_name=label,
                     title_subplots=column_label, 
                     cmap=cmap, 
                     norm=norm)

fig = plt.gcf()

map_axes = list(axes.flat)

cbar_axes = [a for a in fig.axes if a not in map_axes]

for map_ax, cax in zip(map_axes, cbar_axes):
    pcolormesh = map_ax.collections[-1]   # the last collection is the data layer
    cax.clear()
   
    fig.colorbar(pcolormesh, 
                 cax=cax, 
                 spacing="proportional", 
                 boundaries=levels, 
                 ticks=levels, 
                 label=label,
                 extend="max")
    
    cax.tick_params(labelsize=6) 

plt.show()

In [ ]:
############################################################################################################
# return periods
############################################################################################################

In [ ]:
cmap = ListedColormap(["#fff7bc",  # 0–1   pale yellow
                       "#fee391",  # 1–2   yellow
                       "#fec44f",  # 2–3   amber
                       "#fe9929",  # 3–5   orange
                       "#ec7014",  # 5–8   dark orange
                       "#cc4c02",  # 8–12  burnt orange
                       "#a63603",  # 12–17 brick
                       "#7f2704",  # 17–23 dark brick
                       "#67000d"]) # 23–30 deep red
                      
cmap.set_bad("lightgray")

cmap.set_over("#3f0000") 

levels = [0, 1, 2, 3, 5, 8, 12, 17, 23, 30]

norm = BoundaryNorm(levels, ncolors=cmap.N, clip=False)

ib_all_tracks_haz.intensity = sparse.csc_matrix(ib_all_tracks_haz.intensity)
return_periods, label, column_label = ib_all_tracks_haz.local_return_period([10, 15, 20, 25]) # given exceedance intensities
ib_all_tracks_haz.intensity = sparse.csr_matrix(ib_all_tracks_haz.intensity)


# NaN caused by zero exceed_frequency -> infinite (1000000) return period
for threshold in [10, 15, 20, 25]:
    exceed = ib_all_tracks_haz.intensity > threshold

    exceed_frequency = np.asarray(
        exceed.T.dot(ib_all_tracks_haz.frequency)
    ).ravel()

    climada_nan = return_periods[str(threshold)].isna()
    
    return_periods.loc[climada_nan & (exceed_frequency == 0), str(threshold)] = 1000000


# mask People's Republic of China as nan
shapefile = shapereader.natural_earth(resolution="10m", category="cultural", name="admin_0_countries")

world = gpd.read_file(shapefile)

china = world[world.NAME == "China"].geometry.union_all()

china_mask = return_periods.geometry.within(china)

value_cols = [col for col in return_periods.columns if col != "geometry"]

return_periods.loc[china_mask, value_cols] = np.nan

In [ ]:
############################################################################################################
# plot return periods
############################################################################################################

In [ ]:
axes = plot_from_gdf(return_periods, 
                     colorbar_name=label, 
                     title_subplots=column_label, 
                     cmap=cmap, 
                     norm=norm)

fig = plt.gcf()

map_axes = list(axes.flat)

cbar_axes = [a for a in fig.axes if a not in map_axes]

for map_ax, cax in zip(map_axes, cbar_axes):
    pcolormesh = map_ax.collections[-1]   # the last collection is the data layer
    cax.clear()
   
    fig.colorbar(pcolormesh, 
                 cax=cax, 
                 spacing="proportional", 
                 boundaries=levels, 
                 ticks=levels, 
                 label=label, 
                 extend="max")
    
    cax.tick_params(labelsize=6) 

plt.show()

In [ ]:
############################################################################################################
# the "ib_all_tracks_synth" hazard object 
############################################################################################################

In [ ]:
ib_all_tracks.equal_timestep() # must be done before sythetic generation 

# nb_synth_tracks = how many synthetic tracks is computed for every track
# generate synthetic tracks based on directed random walk
ib_all_tracks.calc_perturbed_trajectories(nb_synth_tracks=10) 
                                                           
ib_all_tracks.plot()

In [ ]:
# centroids = from WRF
lat_min, lat_max = 20.0, 29.0
lon_min, lon_max = 117.0, 126.0

files_list = sorted(glob.glob("/lfs/home/yanlan/climada/tc-risk/PGW4K.v230926/wrfout_d01_*_48.nc"))

with xr.open_dataset(files_list[0]) as ref:

    # mask is of xarray.DataArray
    mask = (
            (ref["XLAT"].isel(Time=0)>=lat_min) & (ref["XLAT"].isel(Time=0)<=lat_max) &
            (ref["XLONG"].isel(Time=0)>=lon_min) & (ref["XLONG"].isel(Time=0)<=lon_max)
           ) 

    centroids = Centroids(
                          lat=ref["XLAT"].isel(Time=0).values.ravel()[mask.values.ravel()],
                          lon=ref["XLONG"].isel(Time=0).values.ravel()[mask.values.ravel()]
                         )
############################################################################################################
# other Hazard variables 
# ...

In [ ]:
ib_all_tracks_synth = TropCyclone.from_tracks(tracks=ib_all_tracks, 
                                              centroids=centroids, # centroids = from WRF
                                              model='H1980', 
                                              model_kwargs={"gradient_to_surface_winds": 0.9},
                                              intensity_thres=0) 

In [ ]:
ib_all_tracks_synth.check()

In [ ]:
############################################################################################################
# exceedance_intensities
############################################################################################################

In [ ]:
cmap = ListedColormap(["#fff7bc",  # 0–10   pale yellow
                       "#fee391",  # 10–20  yellow
                       "#fec44f",  # 20–30  amber
                       "#fe9929",  # 30–35  orange
                       "#ec7014",  # 35–40  dark orange
                       "#cc4c02",  # 40–45  burnt orange
                       "#a63603",  # 45–50  brick
                       "#7f2704",  # 50–55  dark brick
                       "#67000d"]) # 55–60  deep red
                      
cmap.set_bad("lightgray")  

cmap.set_over("#3f0000") 

levels = [0, 10, 20, 30, 35, 40, 45, 50, 55, 60]
 
norm = BoundaryNorm(levels, ncolors=cmap.N, clip=False)

ib_all_tracks_synth.intensity = sparse.csc_matrix(ib_all_tracks_synth.intensity)
exceedance_intensities, label, column_label = ib_all_tracks_synth.local_exceedance_intensity([10, 20, 30, 50]) # given return periods
ib_all_tracks_synth.intensity = sparse.csr_matrix(ib_all_tracks_synth.intensity)

# mask People's Republic of China as nan
shapefile = shapereader.natural_earth(resolution="10m", category="cultural", name="admin_0_countries")

world = gpd.read_file(shapefile)

china = world[world.NAME == "China"].geometry.union_all()

china_mask = exceedance_intensities.geometry.within(china)

value_cols = [col for col in exceedance_intensities.columns if col != "geometry"]

exceedance_intensities.loc[china_mask, value_cols] = np.nan

In [ ]:
############################################################################################################
# plot exceedance_intensities
############################################################################################################

In [ ]:
axes = plot_from_gdf(exceedance_intensities,
                     colorbar_name=label,
                     title_subplots=column_label, 
                     cmap=cmap, 
                     norm=norm)

fig = plt.gcf()

map_axes = list(axes.flat)

cbar_axes = [a for a in fig.axes if a not in map_axes]

for map_ax, cax in zip(map_axes, cbar_axes):
    pcolormesh = map_ax.collections[-1]   # the last collection is the data layer
    cax.clear()
   
    fig.colorbar(pcolormesh, 
                 cax=cax, 
                 spacing="proportional", 
                 boundaries=levels, 
                 ticks=levels, 
                 label=label,
                 extend="max")
    
    cax.tick_params(labelsize=6) 

plt.show()

In [ ]:
############################################################################################################
# return periods
############################################################################################################

In [ ]:
cmap = ListedColormap(["#fff7bc",  # 0–1   pale yellow
                       "#fee391",  # 1–2   yellow
                       "#fec44f",  # 2–3   amber
                       "#fe9929",  # 3–5   orange
                       "#ec7014",  # 5–8   dark orange
                       "#cc4c02",  # 8–12  burnt orange
                       "#a63603",  # 12–17 brick
                       "#7f2704",  # 17–23 dark brick
                       "#67000d"]) # 23–30 deep red
                      
cmap.set_bad("lightgray")

cmap.set_over("#3f0000") 

levels = [0, 1, 2, 3, 5, 8, 12, 17, 23, 30]

norm = BoundaryNorm(levels, ncolors=cmap.N, clip=False)

ib_all_tracks_synth.intensity = sparse.csc_matrix(ib_all_tracks_synth.intensity)
return_periods, label, column_label = ib_all_tracks_synth.local_return_period([10, 15, 20, 25]) # given exceedance intensities
ib_all_tracks_synth.intensity = sparse.csr_matrix(ib_all_tracks_synth.intensity)


# NaN caused by zero exceed_frequency -> infinite (1000000) return period
for threshold in [10, 15, 20, 25]:
    exceed = ib_all_tracks_synth.intensity > threshold

    exceed_frequency = np.asarray(
        exceed.T.dot(ib_all_tracks_synth.frequency)
    ).ravel()

    climada_nan = return_periods[str(threshold)].isna()
    
    return_periods.loc[climada_nan & (exceed_frequency == 0), str(threshold)] = 1000000


# mask People's Republic of China as nan
shapefile = shapereader.natural_earth(resolution="10m", category="cultural", name="admin_0_countries")

world = gpd.read_file(shapefile)

china = world[world.NAME == "China"].geometry.union_all()

china_mask = return_periods.geometry.within(china)

value_cols = [col for col in return_periods.columns if col != "geometry"]

return_periods.loc[china_mask, value_cols] = np.nan

In [ ]:
############################################################################################################
# plot return periods
############################################################################################################

In [ ]:
axes = plot_from_gdf(return_periods, 
                     colorbar_name=label, 
                     title_subplots=column_label, 
                     cmap=cmap, 
                     norm=norm)

fig = plt.gcf()

map_axes = list(axes.flat)

cbar_axes = [a for a in fig.axes if a not in map_axes]

for map_ax, cax in zip(map_axes, cbar_axes):
    pcolormesh = map_ax.collections[-1]   # the last collection is the data layer
    cax.clear()
   
    fig.colorbar(pcolormesh, 
                 cax=cax, 
                 spacing="proportional", 
                 boundaries=levels, 
                 ticks=levels, 
                 label=label, 
                 extend="max")
    
    cax.tick_params(labelsize=6) 

plt.show()